# 🗳️ Tamil Nadu Election EDA — 2021 vs 2026
**Codebasics Resume Project Challenge**

A complete, step-by-step EDA notebook answering three research questions:

| # | Question |
|---|----------|
| Q1 | **Geographic story** — Seat distribution shift across TN's 6 regions |
| Q2 | **Flip story** — How many constituencies changed winning party? (Sankey) |
| Q3 | **Vote share story** — Where did TVK's votes come from? |

### ▶️ How to run this notebook
1. Place this notebook in the **same folder** as your three CSV files
2. Run cells **top to bottom**, one section at a time
3. Each section is self-contained with its own explanation

### 📁 Files needed in the same folder
```
tn_2021_results.csv
tn_2026_results.csv
constituency_master.csv
```


---
## ⚙️ Step 0 — Install Libraries
Run this once. Skip if already installed.

In [ ]:
# Run once to install everything needed
# If already installed, this is harmless
!pip install pandas matplotlib seaborn plotly kaleido==0.2.1 --quiet
print("✅ Done")

---
## 📦 Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

# ── Global matplotlib style ──────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "white",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "grid.linestyle":    "--",
    "font.size":         11,
})

print("✅ Libraries ready")

---
## 📂 Step 2 — Load the Data

Three files, all joined on **`ac_number`** (the official ECI constituency ID 1–234).


In [ ]:
df21   = pd.read_csv("tn_2021_results.csv")
df26   = pd.read_csv("tn_2026_results.csv")
master = pd.read_csv("constituency_master.csv")

print(f"2021 → {len(df21):,} rows | {df21['ac_number'].nunique()} constituencies | {df21['party'].nunique()} parties")
print(f"2026 → {len(df26):,} rows | {df26['ac_number'].nunique()} constituencies | {df26['party'].nunique()} parties")
print(f"Master → {len(master)} constituencies")

In [ ]:
df21.head(6)

In [ ]:
df26.head(6)

In [ ]:
master.head(6)

---
## 🔍 Step 3 — Basic EDA (Understand the Data First)

Always explore before analysing. Check dtypes, nulls, distributions.


In [ ]:
# Shape, dtypes, nulls
print("=== 2021 ===")
df21.info()

In [ ]:
print("=== 2026 ===")
df26.info()
# NOTE: 2026 turnout column is intentionally blank (per metadata.txt)
# We derive total votes per AC from candidate vote sums instead

In [ ]:
# How many unique constituencies, parties, candidates per year?
for year, df in [("2021", df21), ("2026", df26)]:
    print(f"--- {year} ---")
    print(f"  Constituencies : {df['ac_number'].nunique()}")
    print(f"  Parties        : {df['party'].nunique()}")
    print(f"  Candidates     : {df['candidate'].nunique()}")
    print(f"  Regions        : {sorted(df['region'].dropna().unique())}")
    print(f"  Reserved cats  : {sorted(df['reserved'].dropna().unique())}")
    print()

In [ ]:
# Basic numeric stats: votes (2021)
df21[["votes", "turnout"]].describe().round(2)

In [ ]:
# Basic numeric stats: votes (2026) — note turnout all NaN
df26[["votes"]].describe().round(2)

In [ ]:
# How many candidates contest each seat?
cands_per_ac_21 = df21[df21["party"] != "NOTA"].groupby("ac_number")["candidate"].count()
cands_per_ac_26 = df26[df26["party"] != "NOTA"].groupby("ac_number")["candidate"].count()

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, data, year, color in zip(
    axes,
    [cands_per_ac_21, cands_per_ac_26],
    ["2021", "2026"],
    ["#457B9D", "#E63946"],
):
    ax.hist(data, bins=range(5, 35), color=color, edgecolor="white", linewidth=0.6)
    ax.axvline(data.mean(), color="black", linestyle="--", linewidth=1.5,
               label=f"Mean: {data.mean():.1f}")
    ax.set_title(f"{year} — Candidates per Constituency", fontsize=12, fontweight="bold")
    ax.set_xlabel("Number of candidates")
    ax.set_ylabel("Number of constituencies")
    ax.legend(fontsize=10)

plt.suptitle("How Contested Is Each Seat?", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 2021 voter turnout distribution (2026 turnout blank in this dataset)
turnout_ac = df21.drop_duplicates("ac_number")[["ac_number", "turnout", "region"]]

fig, ax = plt.subplots(figsize=(11, 4))
ax.hist(turnout_ac["turnout"], bins=28, color="#2A9D8F", edgecolor="white", linewidth=0.6)
ax.axvline(turnout_ac["turnout"].mean(), color="#E63946", linestyle="--", linewidth=2,
           label=f"State mean: {turnout_ac['turnout'].mean():.1f}%")
ax.set_xlabel("Voter turnout (%)", fontsize=12)
ax.set_ylabel("Number of constituencies", fontsize=12)
ax.set_title("2021 Voter Turnout Distribution — 234 Constituencies", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# 2021 turnout by region (box plot)
REGION_ORDER = ["Chennai Metro", "North", "Central", "Kongu", "Delta", "South"]

fig, ax = plt.subplots(figsize=(11, 4))
region_turnout = [
    turnout_ac[turnout_ac["region"] == r]["turnout"].values
    for r in REGION_ORDER
]
bp = ax.boxplot(region_turnout, labels=REGION_ORDER, patch_artist=True,
                medianprops=dict(color="black", linewidth=2))
colors = ["#E63946","#457B9D","#2A9D8F","#F4A261","#6A0572","#E76F51"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel("Turnout (%)", fontsize=11)
ax.set_title("2021 Voter Turnout by Region (box = IQR, line = median)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 🏆 Step 4 — Extract Winners per Constituency

Every downstream analysis needs **one row per constituency** — the winning candidate.

**Logic:** Sort by votes descending within each AC → first row = winner. NOTA excluded (it cannot win a seat).


In [ ]:
def get_winners(df):
    """
    Returns one row per constituency: the candidate with the most votes.
    Also adds: runner_votes, total_votes, margin, winner_share.
    """
    # Exclude NOTA — cannot win a seat
    cands = df[df["party"] != "NOTA"].copy()

    # Sort descending by votes within each constituency
    ranked = cands.sort_values("votes", ascending=False)

    # Winner = first row after sort
    winners = ranked.groupby("ac_number", as_index=False).first()
    winners = winners.rename(columns={"votes": "winner_votes",
                                      "party": "winner_party",
                                      "candidate": "winner_candidate"})

    # Runner-up = second row (for margin calculation)
    runner = (
        ranked.groupby("ac_number", as_index=False).nth(1)
        [["ac_number", "votes"]]
        .rename(columns={"votes": "runner_votes"})
    )

    # Total valid votes per AC (including NOTA) for vote share
    totals = (
        df.groupby("ac_number")["votes"]
        .sum()
        .reset_index(name="total_votes")
    )

    # Merge everything
    result = (
        winners
        .merge(runner,  on="ac_number", how="left")
        .merge(totals,  on="ac_number", how="left")
    )

    # Derived metrics
    result["margin"]       = result["winner_votes"] - result["runner_votes"]
    result["winner_share"] = (result["winner_votes"] / result["total_votes"] * 100).round(2)

    return result


winners_21 = get_winners(df21)
winners_26 = get_winners(df26)

print(f"Winners extracted → 2021: {len(winners_21)} rows | 2026: {len(winners_26)} rows")
winners_21[["ac_number","constituency","region","reserved",
            "winner_party","winner_votes","runner_votes","total_votes","margin","winner_share"]].head()

In [ ]:
# Consistent colour palette — used in ALL charts for readability
PARTY_COLORS = {
    "DMK":    "#E63946",
    "TVK":    "#F4A261",
    "AIADMK": "#2A9D8F",
    "INC":    "#457B9D",
    "PMK":    "#6A0572",
    "VCK":    "#E76F51",
    "BJP":    "#F77F00",
    "CPI":    "#BC4749",
    "CPI(M)": "#8B2635",
    "IUML":   "#2D6A4F",
    "DMDK":   "#B5838D",
    "AMMK":   "#6B705C",
    "NTK":    "#48CAE4",
    "Others": "#ADB5BD",
}

def pcolor(party):
    """Return chart colour for a given party abbreviation."""
    return PARTY_COLORS.get(party, PARTY_COLORS["Others"])

print("✅ Colour palette ready")

In [ ]:
# Overall seat tally: 2021 vs 2026
seats_21 = winners_21["winner_party"].value_counts().rename("seats_2021")
seats_26 = winners_26["winner_party"].value_counts().rename("seats_2026")
seat_tbl  = pd.concat([seats_21, seats_26], axis=1).fillna(0).astype(int)
seat_tbl["net_change"] = seat_tbl["seats_2026"] - seat_tbl["seats_2021"]
seat_tbl = seat_tbl.sort_values("seats_2026", ascending=False)
seat_tbl

In [ ]:
# Bar chart: seat tally comparison
top = seat_tbl[seat_tbl[["seats_2021","seats_2026"]].max(axis=1) >= 1].reset_index()
top.columns = ["party","seats_2021","seats_2026","net_change"]

x, w = np.arange(len(top)), 0.35
fig, ax = plt.subplots(figsize=(13, 5))
b1 = ax.bar(x - w/2, top["seats_2021"], w, label="2021",
            color=[pcolor(p) for p in top["party"]], alpha=0.45, edgecolor="white")
b2 = ax.bar(x + w/2, top["seats_2026"], w, label="2026",
            color=[pcolor(p) for p in top["party"]], edgecolor="white")

for bar in list(b1) + list(b2):
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.5,
                str(int(h)), ha="center", va="bottom", fontsize=8.5)

ax.set_xticks(x)
ax.set_xticklabels(top["party"], fontsize=10)
ax.set_ylabel("Seats won", fontsize=12)
ax.set_title("Tamil Nadu — Overall Seats Won: 2021 vs 2026", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## ❓ Question 1 — The Geographic Story
### How did seat distribution shift across TN's 6 regions between 2021 and 2026?

**Region order used throughout (editorial north-to-south convention):**
`Chennai Metro → North → Central → Kongu → Delta → South`


In [ ]:
# How many seats does each region have? (Constant — no delimitation change)
seats_per_region = (
    master["region"].value_counts()
    .reindex(REGION_ORDER)
    .reset_index()
    .rename(columns={"region": "region", "count": "total_seats"})
)
seats_per_region.columns = ["region", "total_seats"]
seats_per_region

### Q1-A: Pivot table — seats won per party per region

In [ ]:
MAJOR = ["TVK","DMK","AIADMK","INC","PMK","VCK","BJP","IUML","CPI","CPI(M)"]

def group_party(p):
    return p if p in MAJOR else "Others"

def regional_pivot(winners_df):
    df = winners_df.copy()
    df["party_grp"] = df["winner_party"].apply(group_party)
    return (
        df.groupby(["region","party_grp"])
        .size()
        .reset_index(name="seats")
        .pivot(index="region", columns="party_grp", values="seats")
        .reindex(REGION_ORDER)
        .fillna(0)
        .astype(int)
    )

reg21 = regional_pivot(winners_21)
reg26 = regional_pivot(winners_26)

print("=== 2021 Regional Seats ===")
print(reg21.to_string())
print()
print("=== 2026 Regional Seats ===")
print(reg26.to_string())

### Q1-B: Stacked bar chart — regional composition 2021 vs 2026

In [ ]:
STACK_ORDER = ["TVK","DMK","AIADMK","INC","PMK","VCK","BJP","CPI","CPI(M)","IUML","Others"]

fig, axes = plt.subplots(1, 2, figsize=(18, 6), sharey=True)

for ax, (pivot, year) in zip(axes, [(reg21, "2021"), (reg26, "2026")]):
    for col in STACK_ORDER:
        if col not in pivot.columns:
            pivot[col] = 0

    bottom = np.zeros(len(REGION_ORDER))
    x = np.arange(len(REGION_ORDER))

    for party in STACK_ORDER:
        vals = pivot.reindex(REGION_ORDER)[party].fillna(0).values
        if vals.sum() == 0:
            continue
        bars = ax.bar(x, vals, bottom=bottom,
                      color=pcolor(party), label=party,
                      edgecolor="white", linewidth=0.5)
        # Label each segment if big enough
        for bar, v, bot in zip(bars, vals, bottom):
            if v >= 4:
                ax.text(bar.get_x() + bar.get_width()/2, bot + v/2,
                        f"{party}\n{int(v)}",
                        ha="center", va="center",
                        fontsize=7.5, fontweight="bold", color="white")
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(REGION_ORDER, rotation=22, ha="right", fontsize=10)
    ax.set_ylabel("Seats won", fontsize=11)
    ax.set_title(f"{year} — Regional Seat Distribution", fontsize=13, fontweight="bold")
    ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left", framealpha=0.9)

fig.suptitle("Q1: How Each Region Voted — 2021 vs 2026", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### Q1-C: Net seat change heatmap — which party gained/lost in which region?

In [ ]:
all_cols   = sorted(set(reg21.columns) | set(reg26.columns))
reg21_full = reg21.reindex(columns=all_cols, fill_value=0)
reg26_full = reg26.reindex(columns=all_cols, fill_value=0)
change_df  = reg26_full - reg21_full

# Drop columns where nothing changed at all
change_df  = change_df.loc[:, (change_df != 0).any()]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    change_df,
    annot=True, fmt="d",
    cmap="RdYlGn", center=0,
    linewidths=0.5, linecolor="#eee",
    cbar_kws={"label": "Seat change  (2026 − 2021)"},
    ax=ax,
)
ax.set_title("Q1: Net Seat Change per Party per Region (2026 vs 2021)\n"
             "Green = gain  |  Red = loss", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Party", fontsize=11)
ax.set_ylabel("Region", fontsize=11)
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.show()

### Q1-D: Which party dominated each region? (Plotly interactive)

In [ ]:
def long_regional(winners_df, year):
    df = winners_df.copy()
    df["party_grp"] = df["winner_party"].apply(group_party)
    long = (
        df.groupby(["region","party_grp"])
        .size()
        .reset_index(name="seats")
    )
    long["year"] = str(year)
    return long

long_all = pd.concat([long_regional(winners_21, 2021),
                       long_regional(winners_26, 2026)])

fig = px.bar(
    long_all,
    x="region", y="seats",
    color="party_grp", barmode="group",
    facet_col="year",
    color_discrete_map=PARTY_COLORS,
    category_orders={
        "region":    REGION_ORDER,
        "party_grp": MAJOR + ["Others"],
    },
    labels={"seats": "Seats won", "region": "Region", "party_grp": "Party"},
    title="Q1: Regional Seat Distribution — 2021 vs 2026 (interactive: hover for counts)",
    height=500,
)
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white",
                  font=dict(size=12), legend_title="Party")
fig.show()

### Q1-E: Summary table — who led each region?

In [ ]:
def region_leader(winners_df):
    return (
        winners_df
        .groupby(["region","winner_party"])
        .size()
        .reset_index(name="seats")
        .sort_values("seats", ascending=False)
        .groupby("region")
        .first()
        .reset_index()
        [["region","winner_party","seats"]]
    )

lead21 = region_leader(winners_21).rename(columns={"winner_party":"leader_2021","seats":"seats_2021"})
lead26 = region_leader(winners_26).rename(columns={"winner_party":"leader_2026","seats":"seats_2026"})

dom = lead21.merge(lead26, on="region")
dom["changed"] = dom["leader_2021"] != dom["leader_2026"]
dom = dom.set_index("region").reindex(REGION_ORDER).reset_index()
dom

---
## ❓ Question 2 — The Flip Story
### In how many constituencies did the winning party change between 2021 and 2026?


### Q2-A: Build the flip table

In [ ]:
flip = pd.merge(
    winners_21[["ac_number","constituency","region","reserved",
                "winner_party","winner_votes","total_votes","margin"]],
    winners_26[["ac_number","winner_party","winner_votes","total_votes","margin"]],
    on="ac_number",
    suffixes=("_21","_26"),
)
flip["flipped"] = flip["winner_party_21"] != flip["winner_party_26"]

total_flipped  = flip["flipped"].sum()
total_retained = (~flip["flipped"]).sum()

print(f"Total constituencies : 234")
print(f"Retained (same party): {total_retained}  ({total_retained/234*100:.1f}%)")
print(f"Flipped  (new winner): {total_flipped}  ({total_flipped/234*100:.1f}%)")
print()
flip[flip["flipped"]].head(8)

### Q2-B: Flips by region and reservation category

In [ ]:
# By region
flip_region = (
    flip.groupby("region")["flipped"]
    .agg(["sum","count"])
    .rename(columns={"sum":"flipped","count":"total"})
)
flip_region["retained"] = flip_region["total"] - flip_region["flipped"]
flip_region["flip_pct"] = (flip_region["flipped"] / flip_region["total"] * 100).round(1)
flip_region = flip_region.reindex(REGION_ORDER)
print("Flips by Region:")
print(flip_region.to_string())

print()

# By reservation category
flip_res = (
    flip.groupby("reserved")["flipped"]
    .agg(["sum","count"])
    .rename(columns={"sum":"flipped","count":"total"})
)
flip_res["flip_pct"] = (flip_res["flipped"] / flip_res["total"] * 100).round(1)
print("Flips by Reservation Category:")
print(flip_res.to_string())

### Q2-C: Bar chart — retained vs flipped by region

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x, w = np.arange(len(REGION_ORDER)), 0.38

ax.bar(x - w/2, flip_region["retained"], w,
       label="Retained (same party)", color="#2A9D8F", edgecolor="white")
ax.bar(x + w/2, flip_region["flipped"],  w,
       label="Flipped (new winner)",   color="#E63946", edgecolor="white")

# Percentage labels above each red bar
for i, (flipped, pct) in enumerate(zip(flip_region["flipped"], flip_region["flip_pct"])):
    ax.text(i + w/2, flipped + 0.4, f"{pct}%",
            ha="center", va="bottom", fontsize=9.5,
            fontweight="bold", color="#C1121F")

ax.set_xticks(x)
ax.set_xticklabels(REGION_ORDER, fontsize=11)
ax.set_ylabel("Constituencies", fontsize=12)
ax.set_title(
    f"Q2: Seat Flips by Region — {total_flipped} of 234 seats changed hands ({total_flipped/234*100:.0f}%)",
    fontsize=13, fontweight="bold"
)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Q2-D: Sankey diagram — seat flow 2021 → 2026

Each ribbon shows: **from which party won in 2021** → **to which party won in 2026**.  
Ribbon width = number of seats. Hover to see exact counts.


In [ ]:
# Count seats flowing from each 2021 winner → 2026 winner
sankey_data = (
    flip
    .groupby(["winner_party_21","winner_party_26"])
    .size()
    .reset_index(name="seats")
    .sort_values("seats", ascending=False)
)

print("Top seat flows (2021 winner → 2026 winner):")
print(sankey_data.head(15).to_string(index=False))

In [ ]:
# ── Build Sankey node list ────────────────────────────────────────────────
# Left side:  "PartyName '21"   Right side:  "PartyName '26"
left_labels  = [f"{p} '21" for p in sankey_data["winner_party_21"].unique()]
right_labels = [f"{p} '26" for p in sankey_data["winner_party_26"].unique()]

# Deduplicated combined node list preserving order
all_labels, seen = [], set()
for lbl in left_labels + right_labels:
    if lbl not in seen:
        all_labels.append(lbl)
        seen.add(lbl)

node_idx = {lbl: i for i, lbl in enumerate(all_labels)}

sources, targets, values, link_colors = [], [], [], []
for _, row in sankey_data.iterrows():
    src = f"{row['winner_party_21']} '21"
    tgt = f"{row['winner_party_26']} '26"
    if src not in node_idx or tgt not in node_idx:
        continue
    sources.append(node_idx[src])
    targets.append(node_idx[tgt])
    values.append(int(row["seats"]))
    base = pcolor(row["winner_party_21"])
    r, g, b = int(base[1:3],16), int(base[3:5],16), int(base[5:7],16)
    link_colors.append(f"rgba({r},{g},{b},0.45)")

node_colors = [pcolor(lbl.replace(" '21","").replace(" '26","")) for lbl in all_labels]

# ── Draw Sankey ───────────────────────────────────────────────────────────
fig_sk = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(
        pad=18, thickness=24,
        label=all_labels,
        color=node_colors,
        line=dict(color="white", width=0.5),
    ),
    link=dict(
        source=sources, target=targets,
        value=values, color=link_colors,
    ),
))

fig_sk.update_layout(
    title=dict(
        text=(
            f"<b>🔀 Q2: Seat Flow 2021 → 2026</b><br>"
            f"<sup>{total_flipped} of 234 constituencies changed their winning party "
            f"({total_flipped/234*100:.0f}%). Ribbon width = seats.</sup>"
        ),
        x=0.5, xanchor="center", font=dict(size=16),
    ),
    font=dict(size=13),
    paper_bgcolor="white",
    height=620,
    margin=dict(l=30, r=30, t=90, b=30),
)
fig_sk.show()

### Q2-E: Party-level net seat change bar chart

In [ ]:
net = (seats_26.rename("won_26") .to_frame()
       .join(seats_21.rename("won_21"), how="outer")
       .fillna(0).astype(int))
net["change"] = net["won_26"] - net["won_21"]
net = net.sort_values("change")

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#E63946" if c < 0 else "#2A9D8F" for c in net["change"]]
ax.barh(net.index, net["change"], color=colors, edgecolor="white", height=0.7)
ax.axvline(0, color="black", linewidth=0.9)
for i, (party, row) in enumerate(net.iterrows()):
    v = row["change"]
    ax.text(v + (1 if v >= 0 else -1), i, f"{v:+d}",
            va="center", ha="left" if v >= 0 else "right",
            fontsize=10, fontweight="bold")

ax.set_xlabel("Net seat change (2026 − 2021)", fontsize=11)
ax.set_title("Q2: Party-level Net Seat Change — 2021 to 2026",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## ❓ Question 3 — The Vote Share Story
### Where did TVK's votes come from?

**Three-part approach:**
1. State-wide vote share comparison (2021 vs 2026)
2. Regional vote share breakdown
3. Constituency-level correlation — where TVK gained, who lost?


### Q3-A: State-wide vote share — 2021 vs 2026

In [ ]:
def state_vote_share(df, label):
    total = df["votes"].sum()
    share = (
        df.groupby("party")["votes"]
        .sum()
        .div(total)
        .mul(100)
        .round(2)
        .rename(label)
        .sort_values(ascending=False)
    )
    return share

vs21 = state_vote_share(df21, "share_2021")
vs26 = state_vote_share(df26, "share_2026")

vs_compare = pd.concat([vs21, vs26], axis=1).fillna(0)
vs_compare["change_pp"] = (vs_compare["share_2026"] - vs_compare["share_2021"]).round(2)
vs_compare = vs_compare.sort_values("share_2026", ascending=False)

print("State-wide vote share — top parties:")
print(vs_compare[vs_compare[["share_2021","share_2026"]].max(axis=1) >= 1].to_string())

In [ ]:
# Bar chart: state-wide vote share comparison
FOCUS = ["TVK","DMK","AIADMK","INC","NTK","PMK","BJP","AMMK","VCK","NOTA"]
vs_f  = vs_compare.loc[vs_compare.index.isin(FOCUS)].reindex(FOCUS).fillna(0)

x, w = np.arange(len(vs_f)), 0.35
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w/2, vs_f["share_2021"], w, label="2021",
       color=[pcolor(p) for p in vs_f.index], alpha=0.45, edgecolor="white")
ax.bar(x + w/2, vs_f["share_2026"], w, label="2026",
       color=[pcolor(p) for p in vs_f.index], edgecolor="white")

# Change annotation above each pair
for i, (party, row) in enumerate(vs_f.iterrows()):
    chg = row["change_pp"]
    clr = "#E63946" if chg < 0 else "#2A9D8F"
    ypos = max(row["share_2021"], row["share_2026"]) + 0.4
    ax.text(i, ypos, f"{chg:+.1f}", ha="center", va="bottom",
            fontsize=8.5, color=clr, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(vs_f.index, fontsize=11)
ax.set_ylabel("State-wide vote share (%)", fontsize=12)
ax.set_title("Q3: State-wide Party Vote Share — 2021 vs 2026\n"
             "(numbers = change in percentage points)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Q3-B: Regional vote share — TVK vs DMK vs AIADMK across 6 regions

In [ ]:
def regional_vote_share(df):
    region_totals = df.groupby("region")["votes"].sum()
    return (
        df.groupby(["region","party"])["votes"]
        .sum()
        .div(region_totals)
        .mul(100)
        .round(2)
        .reset_index(name="share")
    )

rvs21 = regional_vote_share(df21)
rvs26 = regional_vote_share(df26)

# Build comparison table for key parties
def get_regional_share(rvs, party):
    return (
        rvs[rvs["party"] == party]
        .set_index("region")["share"]
        .reindex(REGION_ORDER)
        .fillna(0)
    )

region_table = pd.DataFrame({
    "TVK_2026":    get_regional_share(rvs26, "TVK"),
    "DMK_2021":    get_regional_share(rvs21, "DMK"),
    "DMK_2026":    get_regional_share(rvs26, "DMK"),
    "AIADMK_2021": get_regional_share(rvs21, "AIADMK"),
    "AIADMK_2026": get_regional_share(rvs26, "AIADMK"),
})
region_table["DMK_change"]    = region_table["DMK_2026"]    - region_table["DMK_2021"]
region_table["AIADMK_change"] = region_table["AIADMK_2026"] - region_table["AIADMK_2021"]

print("Regional vote shares and changes (%):")
print(region_table.round(2).to_string())

In [ ]:
# Line chart: TVK vs DMK vs AIADMK regional share in 2026
x = np.arange(len(REGION_ORDER))

fig, ax = plt.subplots(figsize=(12, 5))
for party, col in [("TVK","TVK_2026"), ("DMK","DMK_2026"), ("AIADMK","AIADMK_2026")]:
    vals = region_table[col].values
    ax.plot(x, vals, marker="o", linewidth=2.4, label=f"{party} 2026", color=pcolor(party))
    ax.plot(x, region_table[col.replace("2026","2021")].values if col != "TVK_2026" else [0]*6,
            marker="s", linewidth=1.4, linestyle="--",
            label=f"{party} 2021" if party != "TVK" else None,
            color=pcolor(party), alpha=0.4)
    for xi, yi in zip(x, vals):
        ax.text(xi, yi + 0.5, f"{yi:.1f}%", ha="center", va="bottom", fontsize=8.5)

ax.set_xticks(x)
ax.set_xticklabels(REGION_ORDER, fontsize=11)
ax.set_ylabel("Vote share in region (%)", fontsize=12)
ax.set_title("Q3: TVK vs DMK vs AIADMK — Regional Vote Share 2026 (solid) vs 2021 (dashed)",
             fontsize=12, fontweight="bold")
handles, labels = ax.get_legend_handles_labels()
ax.legend([h for h,l in zip(handles,labels) if l], [l for l in labels if l],
          fontsize=9, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Grouped bar: how did DMK and AIADMK vote share change per region?
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, (party, col) in zip(axes, [("DMK","DMK_change"), ("AIADMK","AIADMK_change")]):
    vals   = region_table[col].values
    colors = ["#E63946" if v < 0 else "#2A9D8F" for v in vals]
    bars   = ax.bar(REGION_ORDER, vals, color=colors, edgecolor="white", width=0.6)
    ax.axhline(0, color="black", linewidth=0.9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                v + (0.2 if v >= 0 else -0.5),
                f"{v:+.1f}", ha="center",
                va="bottom" if v >= 0 else "top",
                fontsize=9.5, fontweight="bold")
    ax.set_title(f"{party} vote share change 2021→2026 by region\n(red = lost votes, green = gained)",
                 fontsize=12, fontweight="bold")
    ax.set_ylabel("Change in vote share (pp)", fontsize=11)
    ax.tick_params(axis="x", rotation=20)

fig.suptitle("Q3: Where did DMK and AIADMK lose votes?", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### Q3-C: Constituency-level correlation analysis

**Key question:** In constituencies where TVK got more votes in 2026, did DMK or AIADMK lose more?

A **negative correlation** between TVK's 2026 share and a party's vote change means:  
→ Where TVK did well, that party lost votes (TVK pulled from them)


In [ ]:
# Compute per-AC vote share for each party
def ac_vote_shares(df, parties):
    """
    Returns DataFrame: index=ac_number, columns=party names, values=% of total votes in that AC.
    """
    ac_totals = df.groupby("ac_number")["votes"].sum()
    result = {}
    for party in parties:
        pv = df[df["party"] == party].groupby("ac_number")["votes"].sum()
        result[party] = (pv.div(ac_totals) * 100).round(3)
    return pd.DataFrame(result).fillna(0)

PARTIES_21 = ["DMK","AIADMK","INC","PMK","NTK","VCK","BJP"]
PARTIES_26 = ["TVK","DMK","AIADMK","INC","NTK"]

ac_21 = ac_vote_shares(df21, PARTIES_21).add_suffix("_21")
ac_26 = ac_vote_shares(df26, PARTIES_26).add_suffix("_26")

ac_all = ac_21.join(ac_26, how="inner")
ac_all = ac_all.merge(master[["ac_number","region","reserved"]], on="ac_number", how="left")

# Change columns
ac_all["DMK_chg"]    = ac_all["DMK_26"]    - ac_all["DMK_21"]
ac_all["AIADMK_chg"] = ac_all["AIADMK_26"] - ac_all["AIADMK_21"]

print(f"AC-level dataset: {len(ac_all)} constituencies")
print()

# Correlations
for party, col in [("DMK","DMK_chg"), ("AIADMK","AIADMK_chg")]:
    r = ac_all["TVK_26"].corr(ac_all[col])
    print(f"Correlation: TVK 2026 share vs {party} change = {r:+.3f}")
print()
print("Interpretation: negative = where TVK gained, that party lost votes")

In [ ]:
# Scatter: TVK 2026 vs DMK change and AIADMK change, coloured by region
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, (party, col) in zip(axes, [("DMK","DMK_chg"), ("AIADMK","AIADMK_chg")]):
    r = ac_all["TVK_26"].corr(ac_all[col])
    for region in REGION_ORDER:
        sub = ac_all[ac_all["region"] == region]
        ax.scatter(sub["TVK_26"], sub[col], alpha=0.5, s=35, label=region)

    # Trend line
    m, b = np.polyfit(ac_all["TVK_26"], ac_all[col], 1)
    xr = np.linspace(ac_all["TVK_26"].min(), ac_all["TVK_26"].max(), 100)
    ax.plot(xr, m*xr + b, color="black", linewidth=1.5, linestyle="--")

    ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")
    ax.set_xlabel("TVK vote share in 2026 (%)", fontsize=11)
    ax.set_ylabel(f"{party} vote share change  2021→2026 (pp)", fontsize=11)
    ax.set_title(f"TVK 2026 vs {party} change  (r = {r:+.3f})", fontsize=12, fontweight="bold")
    ax.legend(fontsize=8, loc="lower left")

fig.suptitle("Q3: Where TVK Gained — Which Party Lost?\nEach dot = one constituency",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### Q3-D: Plotly interactive scatter (hover to see constituency name)

In [ ]:
corr_dmk    = ac_all["TVK_26"].corr(ac_all["DMK_chg"])
corr_aiadmk = ac_all["TVK_26"].corr(ac_all["AIADMK_chg"])

fig = px.scatter(
    ac_all.reset_index(),
    x="TVK_26", y="DMK_chg",
    color="region",
    hover_data={
        "ac_number":   True,
        "region":      True,
        "TVK_26":     ":.1f",
        "DMK_chg":    ":.1f",
        "AIADMK_chg": ":.1f",
    },
    trendline="ols",
    labels={
        "TVK_26":  "TVK vote share 2026 (%)",
        "DMK_chg": "DMK vote share change 2021→2026 (pp)",
        "region":  "Region",
    },
    title=f"Q3: TVK 2026 vs DMK change (r = {corr_dmk:+.3f}) — hover for constituency",
    height=520,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)
fig.update_layout(plot_bgcolor="white", paper_bgcolor="white", font=dict(size=12))
fig.show()

In [ ]:
fig2 = px.scatter(
    ac_all.reset_index(),
    x="TVK_26", y="AIADMK_chg",
    color="region",
    hover_data={
        "ac_number":   True,
        "region":      True,
        "TVK_26":     ":.1f",
        "DMK_chg":    ":.1f",
        "AIADMK_chg": ":.1f",
    },
    trendline="ols",
    labels={
        "TVK_26":      "TVK vote share 2026 (%)",
        "AIADMK_chg":  "AIADMK vote share change 2021→2026 (pp)",
        "region":      "Region",
    },
    title=f"Q3: TVK 2026 vs AIADMK change (r = {corr_aiadmk:+.3f}) — hover for constituency",
    height=520,
)
fig2.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)
fig2.update_layout(plot_bgcolor="white", paper_bgcolor="white", font=dict(size=12))
fig2.show()

### Q3-E: Quadrant analysis
Classify every constituency by what happened to DMK and AIADMK when TVK appeared.

| Quadrant | DMK | AIADMK | Implication |
|---|---|---|---|
| Both dropped | ↓ | ↓ | TVK likely pulled from both |
| Only DMK dropped | ↓ | ↑ | TVK mainly pulled from DMK |
| Only AIADMK dropped | ↑ | ↓ | TVK mainly pulled from AIADMK |
| Both grew | ↑ | ↑ | TVK may have mobilised new voters |


In [ ]:
ac_all["quadrant"] = "Both grew"
ac_all.loc[(ac_all["DMK_chg"] < 0) & (ac_all["AIADMK_chg"] < 0), "quadrant"] = "Both dropped"
ac_all.loc[(ac_all["DMK_chg"] < 0) & (ac_all["AIADMK_chg"] > 0), "quadrant"] = "Only DMK dropped"
ac_all.loc[(ac_all["DMK_chg"] > 0) & (ac_all["AIADMK_chg"] < 0), "quadrant"] = "Only AIADMK dropped"

quad = ac_all["quadrant"].value_counts().reset_index()
quad.columns = ["quadrant","constituencies"]
quad["pct"] = (quad["constituencies"] / 234 * 100).round(1)
print(quad.to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
q_colors = {"Both dropped": "#E63946",
             "Only DMK dropped": "#457B9D",
             "Only AIADMK dropped": "#2A9D8F",
             "Both grew": "#ADB5BD"}
wedges, texts, autotexts = ax1.pie(
    quad["constituencies"],
    labels=quad["quadrant"],
    autopct="%1.1f%%",
    colors=[q_colors[q] for q in quad["quadrant"]],
    startangle=140,
    wedgeprops=dict(edgecolor="white", linewidth=1.5),
)
for t in texts:    t.set_fontsize(9.5)
for a in autotexts: a.set_fontsize(9.5); a.set_fontweight("bold")
ax1.set_title("Q3: In how many ACs did\nboth parties drop vs just one?",
              fontsize=12, fontweight="bold")

# Scatter with quadrant colouring
colors_map = {
    "Both dropped":        "#E63946",
    "Only DMK dropped":    "#457B9D",
    "Only AIADMK dropped": "#2A9D8F",
    "Both grew":           "#ADB5BD",
}
for q, grp in ac_all.groupby("quadrant"):
    ax2.scatter(grp["TVK_26"], grp["DMK_chg"],
                c=colors_map[q], alpha=0.55, s=30, label=q)
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.axvline(ac_all["TVK_26"].mean(), color="gray", linewidth=0.8, linestyle=":")
ax2.set_xlabel("TVK vote share 2026 (%)", fontsize=11)
ax2.set_ylabel("DMK vote share change (pp)", fontsize=11)
ax2.set_title("Quadrant view — DMK change vs TVK gain", fontsize=12, fontweight="bold")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

### Q3-F: Final summary — headline numbers for your slide

In [ ]:
tvk_share  = vs26.get("TVK", 0)
dmk_21_s   = vs21.get("DMK", 0)
dmk_26_s   = vs26.get("DMK", 0)
ai_21_s    = vs21.get("AIADMK", 0)
ai_26_s    = vs26.get("AIADMK", 0)

q_both  = ac_all[ac_all["quadrant"] == "Both dropped"].shape[0]
q_dmk   = ac_all[ac_all["quadrant"] == "Only DMK dropped"].shape[0]
q_ai    = ac_all[ac_all["quadrant"] == "Only AIADMK dropped"].shape[0]
q_new   = ac_all[ac_all["quadrant"] == "Both grew"].shape[0]

print("=" * 62)
print("  KEY HEADLINE NUMBERS FOR YOUR DECK")
print("=" * 62)
print()
print("Q1 — GEOGRAPHIC STORY")
print(f"  Check the heatmap: which region shifted most dramatically")
print()
print("Q2 — FLIP STORY")
print(f"  {total_flipped} of 234 seats ({total_flipped/234*100:.0f}%) changed winning party")
print(f"  {total_retained} seats retained the same winning party")
print()
print("Q3 — VOTE SHARE STORY")
print(f"  TVK state-wide vote share (2026)    : {tvk_share:.2f}%")
print(f"  DMK  2021 → 2026                    : {dmk_21_s:.2f}% → {dmk_26_s:.2f}%  ({dmk_26_s-dmk_21_s:+.2f} pp)")
print(f"  AIADMK 2021 → 2026                  : {ai_21_s:.2f}% → {ai_26_s:.2f}%  ({ai_26_s-ai_21_s:+.2f} pp)")
print()
print(f"  Correlation TVK vs DMK change       : {corr_dmk:+.3f}")
print(f"  Correlation TVK vs AIADMK change    : {corr_aiadmk:+.3f}")
print()
print(f"  ACs where BOTH DMK+AIADMK dropped   : {q_both} ({q_both/234*100:.0f}%)")
print(f"  ACs where ONLY DMK dropped          : {q_dmk}  ({q_dmk/234*100:.0f}%)")
print(f"  ACs where ONLY AIADMK dropped       : {q_ai}  ({q_ai/234*100:.0f}%)")
print(f"  ACs where BOTH grew                 : {q_new}  ({q_new/234*100:.0f}%)")
print()
print("=" * 62)